In [5]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from model import GraphSAGE
import copy

In [8]:
checkpoint = torch.load('models/task_a_model.pt', weights_only=False)
data = checkpoint['data']
student_id_map = checkpoint['student_id_map']
course_id_map = checkpoint['course_id_map']
concept_id_map = checkpoint['concept_id_map']

model = GraphSAGE(hidden_dim=256)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

students = pd.read_csv('datasets/students.csv')
courses = pd.read_csv('datasets/courses.csv')
enrollments = pd.read_csv('datasets/enrollments.csv')
chatbot = pd.read_csv('datasets/chatbot_signals.csv')

course_name_map = courses.set_index('course_id')['course_name'].to_dict()

print(f"Current student count: {data['student'].x.shape[0]}")

Current student count: 2000


In [9]:
sim_data = copy.deepcopy(data)

In [10]:
new_gpa = 3.0
avg_effort = students['effort_level'].mean()
avg_risk = students['risk_factor'].mean()
avg_confusion = 0.0
avg_questions = 0.0

existing_features = sim_data['student'].x

print("Existing student features shape:", existing_features.shape)
print("Feature means:", existing_features.mean(dim=0))
print("Feature stds:", existing_features.std(dim=0))

Existing student features shape: torch.Size([2000, 5])
Feature means: tensor([ 4.7684e-10, -2.8610e-09, -4.2915e-09, -9.5367e-10, -1.4305e-09])
Feature stds: tensor([1.0003, 1.0003, 1.0003, 1.0003, 1.0003])


In [11]:
raw_means = {
    'gpa_start': students['gpa_start'].mean(),
    'effort_level': students['effort_level'].mean(),
    'risk_factor': students['risk_factor'].mean(),
    'avg_confusion': chatbot.groupby('student_id')['confusion_score'].mean().mean(),
    'avg_questions': chatbot.groupby('student_id')['num_questions'].mean().mean()
}

raw_stds = {
    'gpa_start': students['gpa_start'].std(),
    'effort_level': students['effort_level'].std(),
    'risk_factor': students['risk_factor'].std(),
    'avg_confusion': chatbot.groupby('student_id')['confusion_score'].mean().std(),
    'avg_questions': chatbot.groupby('student_id')['num_questions'].mean().std()
}

new_features = torch.tensor([[
    (new_gpa - raw_means['gpa_start']) / raw_stds['gpa_start'],
    (avg_effort - raw_means['effort_level']) / raw_stds['effort_level'],
    (avg_risk - raw_means['risk_factor']) / raw_stds['risk_factor'],
    (avg_confusion - raw_means['avg_confusion']) / raw_stds['avg_confusion'],
    (avg_questions - raw_means['avg_questions']) / raw_stds['avg_questions']
]], dtype=torch.float)

print("New student normalized features:", new_features)

New student normalized features: tensor([[ 1.5649,  0.0000,  0.0000, -2.3496, -3.6744]])
